In [64]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [65]:
data_gdynia = pd.read_csv(
    "../../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

A,B

In [66]:
data_gdynia.resample("D", on="date_of_checking").size().to_frame(
    name="num_buildings"
).assign(
    num_buildings_cum=lambda d: d["num_buildings"].cumsum(),
    share_cum=lambda d: d["num_buildings_cum"].div(len(data_gdynia)),
    shift_cum=lambda d: d["num_buildings"].diff(1),
).assign(
    shift_cum=lambda d: d["shift_cum"].where(d["shift_cum"].notna(), d["num_buildings"])
)

,num_buildings,num_buildings_cum,share_cum,shift_cum
date_of_checking,,,,
2026-09-23,52,52,0.052953,52.0
2026-09-24,108,160,0.162933,56.0
2026-09-25,67,227,0.231161,-41.0
2026-09-26,100,327,0.332994,33.0
2026-09-27,461,788,0.802444,361.0
2026-09-28,0,788,0.802444,-461.0
2026-09-29,22,810,0.824847,22.0
2026-09-30,172,982,1.000000,150.0


W 1 wierszu zdecydowałem się wrzucić 1 wartość num build, gdyż zmiana od 0 to 52.
Zmiany w okresie mówią prawde i nie mówią gdyż nie prowadziłem inwentaryzacji tego dnia, lecz zmiany względem dni się zgadzają, a względem dni inwentaryzacji nie, wiec to jest ambiwalentny element

C.D

In [ ]:
date_of_check_streets = (
    data_gdynia.groupby(["date_of_checking", "addr_street"])
    .size()
    .to_frame("num_build")
    .assign(
        rank_per_date=lambda d: d.groupby("date_of_checking")["num_build"].rank(
            "min", ascending=False
        )
    )
    .loc[lambda d: d["rank_per_date"].le(3)]
    .sort_values(["date_of_checking", "rank_per_date"], ascending=True)
    .join(
        data_gdynia.groupby("addr_street")["date_of_checking"].min(), on="addr_street"
    )
    .rename(columns={"date_of_checking": "first_date_of_checking"})
    .reset_index()
    .assign(is_first_day=lambda d: d["date_of_checking"] == d["first_date_of_checking"])
    .set_index(["date_of_checking", "addr_street"])
)

date_of_check_streets.head()

num_build  rank_per_date  \
date_of_checking addr_street                                          
2026-09-23       Starowiejska                     17            1.0   
                 3 Maja                            6            2.0   
                 Antoniego Hryniewickiego          5            3.0   
                 Mściwoja                          5            3.0   
2026-09-24       10 Lutego                        24            1.0   

                                          first_date_of_checking  is_first_day  
date_of_checking addr_street                                                    
2026-09-23       Starowiejska                         2026-09-23          True  
                 3 Maja                               2026-09-23          True  
                 Antoniego Hryniewickiego             2026-09-23          True  
                 Mściwoja                             2026-09-23          True  
2026-09-24       10 Lutego                            2026-09-24          True

In [ ]:
print(date_of_check_streets["is_first_day"].sum())
print(len(date_of_check_streets))
print(date_of_check_streets.reset_index().loc[:, "addr_street"].nunique())

10
27
16


In [ ]:
date_of_check_streets.groupby("date_of_checking")["is_first_day"].sum()

date_of_checking
2026-09-23    4
2026-09-24    2
2026-09-25    2
2026-09-26    0
2026-09-27    0
2026-09-29    1
2026-09-30    1
Name: is_first_day, dtype: int64

Remisy zostały obsłużone raank min, czyli jeżeli są remisy na 3 miejscu to jest więcej wartości jak na 1 czy 2 to 3
Mamy długość 27 ulic, czyli na każdy z 7 dni, średnio po prawie 4, z czego 16 unikalnych i 10 z tych 16 w date of checking miało swój 1 dzień inwentaryzacji, mówi to że inwentaryzacja szła ulicami